# Modelos n-grama

In [111]:
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jarat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jarat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1.1 Exploración del dataset

In [2]:
import datasets 

from datasets import load_dataset

ds = load_dataset("jhonrayo99/nlp-tarea-2-ngramas")

d:\backup\MAESTRIA\MINE\2026_2\ProcesamientoLenguajeNatural\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jarat\.cache\huggingface\hub\datasets--jhonrayo99--nlp-tarea-2-ngramas. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 28779/28779 [00:00<00:00, 41405

In [12]:
print(ds)

for nombre in ds.keys():
    print(f"\n--- {nombre} ---")
    print(ds[nombre])
    print("Columnas:", ds[nombre].column_names)
    print("Primer ejemplo:")
    print(ds[nombre][0])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 637938
    })
    test: Dataset({
        features: ['text'],
        num_rows: 28779
    })
})

--- train ---
Dataset({
    features: ['text'],
    num_rows: 637938
})
Columnas: ['text']
Primer ejemplo:
{'text': 'LADY SUSAN'}

--- test ---
Dataset({
    features: ['text'],
    num_rows: 28779
})
Columnas: ['text']
Primer ejemplo:
{'text': 'Emma Woodhouse, handsome, clever, and rich, with a comfortable home and'}


La partición de train se realiza a partir del nombre del autor

In [45]:
for i, fila in enumerate(ds["train"]):
    if "JANE AUSTEN" in fila["text"].upper():
        print("Jane Austen encontrada en train en la fila:", i)
        break

Jane Austen encontrada en train en la fila: 4


In [46]:
for i, fila in enumerate(ds["train"]):
    if "MARK TWAIN" in fila["text"].upper():
        print("Mark Twain encontrado en train en la fila:", i)
        break

Mark Twain encontrado en train en la fila: 99888


Al revisar el conjunto de test y no evidenciar el nombre de nos autores, fue necesario revisar hugging face para definir cómo realizar el corte del dataset.

Se identifica que """" comienza "“Camelot--Camelot,” said I to myself. “I don't seem to remember"

In [82]:
for i, fila in enumerate(ds["test"]):
    if "JANE AUSTEN" in fila["text"].upper():
        print("Jane Austen encontrada en test en la fila:", i)
        break
# comienza el test de Jane Austen
for i, fila in enumerate(ds["test"]):
    if "EMMA" in fila["text"].upper():
        print("EMMA encontrada en test en la fila:", i)
        break    

# comienza el test de Mark twein
for i, fila in enumerate(ds["test"]):
    if "Camelot--Camelot" in fila["text"]:
        print("Camelot--Camelot encontrada en test en la fila:", i)
        break    

EMMA encontrada en test en la fila: 0
Camelot--Camelot encontrada en test en la fila: 16293


## 1.2 Separacion de autores

El dataset no tiene una columna que indique el autor. Por eso usamos el orden de los fragmentos, despues de inspeccionar sus encabezados y limites. En entrenamiento, los textos de Jane Austen aparecen primero y Mark Twain comienza en la fila 99888. En prueba, Emma ocupa las filas iniciales y Mark Twain comienza en la fila 16293.

Estos indices son limites del dataset, no indices de oraciones ni de palabras. No mezclaremos los autores porque cada autor debe tener su propio vocabulario y sus propios modelos.

In [83]:
textos_train = ds["train"]["text"]
textos_test = ds["test"]["text"]

# Estos limites fueron identificados durante la inspeccion del dataset.
inicio_twain_train = 99888
inicio_twain_test = 16293

austen_train = textos_train[:inicio_twain_train]
twain_train = textos_train[inicio_twain_train:]

austen_test = textos_test[:inicio_twain_test]
twain_test = textos_test[inicio_twain_test:]

print("Fragmentos de Austen en train:", len(austen_train))
print("Fragmentos de Twain en train:", len(twain_train))
print("Fragmentos de Austen en test:", len(austen_test))
print("Fragmentos de Twain en test:", len(twain_test))

print("\nLimite train:")
print("Ultimo fragmento de Austen:", repr(austen_train[-1]))
print("Primer fragmento de Twain:", repr(twain_train[0]))

print("\nLimite test:")
print("Ultimo fragmento de Emma:", repr(austen_test[-1]))
print("Primer fragmento de Twain:", repr(twain_test[0]))

Fragmentos de Austen en train: 99888
Fragmentos de Twain en train: 538050
Fragmentos de Austen en test: 16293
Fragmentos de Twain en test: 12486

Limite train:
Ultimo fragmento de Austen: ''
Primer fragmento de Twain: 'By Mark Twain'

Limite test:
Ultimo fragmento de Emma: 'ceremony, were fully answered in the perfect happiness of the union.'
Primer fragmento de Twain: "“Camelot--Camelot,” said I to myself. “I don't seem to remember"


## 1.3 Segmentacion, tokenizacion y normalizacion

Los registros del dataset son fragmentos de libros, no necesariamente oraciones completas. Primero unimos los fragmentos de cada autor conservando saltos de linea y luego usamos `sent_tokenize` para detectar los limites de las oraciones. Despues usamos `word_tokenize` para separar cada oracion en palabras y signos de puntuacion.

Normalizamos convirtiendo el texto a minusculas y reemplazando los numeros por `<NUM>`. No eliminamos palabras de parada, no aplicamos stemming y no aplicamos lematizacion. El entrenamiento y la prueba se mantienen separados desde ahora.

In [104]:
import re
from nltk.tokenize import word_tokenize

def preparar_texto(lineas):
    """Limpia fragmentos, segmenta oraciones, normaliza y tokeniza el texto.
    
    Parameters
    ----------------------
        lineas: list[str]
        Lineas de texto a procesar
    Returns
    ----------------------
        oraciones_tokenizadas: list[list[str]]
        Lista de oraciones, donde cada oracion es una lista de tokens.
    """
    lineas_limpias = []

    # 1. Eliminar lineas vacias y espacios exteriores.
    for linea in lineas:
        if linea.strip() != "":
            lineas_limpias.append(linea.strip())
    
    # 2. Unir los fragmentos antes de segmentar.
    texto = "\n".join(lineas_limpias)
    
    # 3. Segmentar en oraciones.
    oraciones = sent_tokenize(texto)
    
    oraciones_tokenizadas = []
    
    for oracion in oraciones:
        # 4. Normalizar: convertir a minusculas.
        oracion = oracion.lower()

        # 5. Separar palabras y signos de puntuacion.
        tokens = word_tokenize(oracion)

        # 6. Reemplazar cada numero por un unico token especial.
        tokens_normalizados = []
        for token in tokens:
            if re.fullmatch(r"\d+(?:[.,]\d+)*", token):
                tokens_normalizados.append("<NUM>")
            else:
                tokens_normalizados.append(token)

        oraciones_tokenizadas.append(tokens_normalizados)
    
    return oraciones_tokenizadas

In [113]:
## Aplicación de la función en las listas train, test de cada autor

austen_train_oraciones = preparar_texto(austen_train)
austen_test_oraciones = preparar_texto(austen_test)

twain_train_oraciones = preparar_texto(twain_train)
twain_test_oraciones = preparar_texto(twain_test)


In [119]:
for i, elem in enumerate(austen_train_oraciones[:2]):
    print(f"Índice {i}: {elem}")

Índice 0: ['lady', 'susan', 'by', 'jane', 'austen', 'contents', 'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii', 'xiii', 'xiv', 'xv', 'xvi', 'xvii', 'xviii', 'xix', 'xx', 'xxi', 'xxii', 'xxiii', 'xxiv', 'xxv', 'xxvi', 'xxvii', 'xxviii', 'xxix', 'xxx', 'xxxi', 'xxxii', 'xxxiii', 'xxxiv', 'xxxv', 'xxxvi', 'xxxvii', 'xxxviii', 'xxxix', 'xl', 'xli', 'conclusion', 'i', '_lady', 'susan', 'vernon', 'to', 'mr.', 'vernon._', 'langford', ',', 'dec.', 'my', 'dear', 'brother', ',', '—', 'i', 'can', 'no', 'longer', 'refuse', 'myself', 'the', 'pleasure', 'of', 'profiting', 'by', 'your', 'kind', 'invitation', 'when', 'we', 'last', 'parted', 'of', 'spending', 'some', 'weeks', 'with', 'you', 'at', 'churchhill', ',', 'and', ',', 'therefore', ',', 'if', 'quite', 'convenient', 'to', 'you', 'and', 'mrs.', 'vernon', 'to', 'receive', 'me', 'at', 'present', ',', 'i', 'shall', 'hope', 'within', 'a', 'few', 'days', 'to', 'be', 'introduced', 'to', 'a', 'sister', 'whom', 'i', 'have', 'so'

In [106]:
# Comparacion usando exactamente las mismas lineas antes y despues del procesamiento.

fragmentos_ejemplo = austen_train[100:150]

print("ANTES, fragmentos originales:")
for fragmento in fragmentos_ejemplo:
    if fragmento.strip() != "":
        print(repr(fragmento))

oraciones_ejemplo = preparar_texto(fragmentos_ejemplo)

print("\nDESPUES, oraciones tokenizadas y normalizadas:")
for tokens in oraciones_ejemplo:
    print(tokens)

ANTES, fragmentos originales:
' CONCLUSION'
'I'
'_Lady Susan Vernon to Mr. Vernon._'
'Langford, Dec.'
'MY DEAR BROTHER,—I can no longer refuse myself the pleasure of'
'profiting by your kind invitation when we last parted of spending some'
'weeks with you at Churchhill, and, therefore, if quite convenient to'
'you and Mrs. Vernon to receive me at present, I shall hope within a few'
'days to be introduced to a sister whom I have so long desired to be'
'acquainted with. My kind friends here are most affectionately urgent'
'with me to prolong my stay, but their hospitable and cheerful'
'dispositions lead them too much into society for my present situation'
'and state of mind; and I impatiently look forward to the hour when I'
'shall be admitted into your delightful retirement.'

DESPUES, oraciones tokenizadas y normalizadas:
['conclusion', 'i', '_lady', 'susan', 'vernon', 'to', 'mr.', 'vernon._', 'langford', ',', 'dec.', 'my', 'dear', 'brother', ',', '—', 'i', 'can', 'no', 'longer', 'refu

## 1.4 Construccion del vocabulario y reemplazo por `<UNK>`

Un vocabulario es el conjunto de tokens que el modelo reconoce. Se construye usando solamente las oraciones de entrenamiento, porque el conjunto de prueba debe representar texto no visto por el modelo.

En este notebook conservamos los tokens cuya frecuencia es mayor que uno. Esto incluye palabras y signos de puntuacion, porque ambos forman parte de las secuencias que usara el modelo. `<UNK>` se agrega explicitamente al vocabulario para representar cualquier token que no sea conocido. Austen y Twain tienen vocabularios separados.

In [ ]:
def construir_vocabulario(oraciones):
    """
    Esta función genera un vocabulario a partir de las palabras que tienen mas de una aparición en las oraciones. 

    Parameters
    -----------
        oraciones: lista de listas de tokens.
    
    Returns
    ------------
        tokens_unicos: Arreglo de tokens que incluye los tokens con frecuencia = 1.
        frecuencias: Arreglo de conteos por token.
        vocabulario: set de strings con el vocabulario depurado.
    """

    # Unir todos los tokens de las oraciones en una sola lista.
    tokens = []

    for oracion in oraciones:
        for token in oracion:
            tokens.append(token)

    # Obtener tokens únicos y sus frecuencias.
    tokens_unicos, frecuencias = np.unique(
        tokens,
        return_counts=True
    )

    # Conservar solamente los tokens con frecuencia mayor que 1.
    vocabulario = set()

    for token, frecuencia in zip(tokens_unicos, frecuencias):
        if frecuencia > 1:
            vocabulario.add(token)

    # <UNK> representa los tokens que el modelo no conoce.
    vocabulario.add("<UNK>")

    return tokens_unicos, frecuencias, vocabulario

In [121]:
## Construcción de los vocabularios 

tokens_austen, frecuencias_austen, vocabulario_austen = construir_vocabulario(austen_train_oraciones)

tokens_twain, frecuencias_twain, vocabulario_twain = construir_vocabulario(twain_train_oraciones)

In [127]:
print("tokens_austen")
print(f"Tipo: {type(tokens_austen)}")
print(f"Primeros elementos: {tokens_austen[:10]}")

print("\nfrecuencias_austen")
print(f"Tipo: {type(frecuencias_austen)}")
print(f"Contenido: {frecuencias_austen}")

print("\nvocabulario_austen")
print(f"Tipo: {type(vocabulario_austen)}")
print(f"Primeros elementos: {list(vocabulario_austen)[:10]}")

tokens_austen
Tipo: <class 'numpy.ndarray'>
Primeros elementos: ['!' '&' "'" "''" "'s" '(' ')' '*' ',' '--']

frecuencias_austen
Tipo: <class 'numpy.ndarray'>
Contenido: [2716  800   33 ... 4287 7333 7263]

vocabulario_austen
Tipo: <class 'set'>
Primeros elementos: [np.str_('peice'), np.str_('dares'), np.str_('boat'), np.str_('roomy'), np.str_('perseveres'), np.str_('improbable'), np.str_('familiarity'), np.str_('possibilities'), np.str_('woodston'), np.str_('ecstatic')]


In [135]:
# ajustar el train para que las palabras con frecuencia 1 sean remplazadas por <UNK>

def reemplazar_unk(oraciones, vocabulario):
    """
    Esta función remplaza los tokens no vistos por <UNK>, para ello usa el vocabulario construido. 

    Parameters
    -----------
        oraciones: lista de listas de tokens.
        vocabulario: set de palabras 
    Returns
    ------------
        oracions_unk: lista de listas de tokens sin palabras desconocidas o con frecuencia menor a 1. Cuenta con tokens de tipo <UNK>
    """
    oraciones_unk = []

    for oracion in oraciones:
        nueva_oracion = []

        for token in oracion:
            if token in vocabulario:
                nueva_oracion.append(token)
            else:
                nueva_oracion.append("<UNK>")

        oraciones_unk.append(nueva_oracion)

    return oraciones_unk

In [145]:
austen_train_unk = reemplazar_unk(austen_train_oraciones,vocabulario_austen)

twain_train_unk = reemplazar_unk(twain_train_oraciones,vocabulario_twain)

austen_test_unk = reemplazar_unk(austen_test_oraciones,vocabulario_austen)

twain_test_unk = reemplazar_unk(twain_test_oraciones,vocabulario_twain)

In [144]:
for i, elem in enumerate(austen_train_unk[0:10]):
    print(f"Índice {i}: {elem}")

Índice 0: ['lady', 'susan', 'by', 'jane', 'austen', 'contents', 'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii', 'xiii', 'xiv', 'xv', 'xvi', 'xvii', 'xviii', 'xix', 'xx', 'xxi', 'xxii', 'xxiii', 'xxiv', 'xxv', 'xxvi', 'xxvii', 'xxviii', 'xxix', 'xxx', 'xxxi', 'xxxii', 'xxxiii', 'xxxiv', 'xxxv', 'xxxvi', 'xxxvii', 'xxxviii', 'xxxix', 'xl', 'xli', 'conclusion', 'i', '_lady', 'susan', 'vernon', 'to', 'mr.', 'vernon._', 'langford', ',', 'dec.', 'my', 'dear', 'brother', ',', '—', 'i', 'can', 'no', 'longer', 'refuse', 'myself', 'the', 'pleasure', 'of', 'profiting', 'by', 'your', 'kind', 'invitation', 'when', 'we', 'last', 'parted', 'of', 'spending', 'some', 'weeks', 'with', 'you', 'at', 'churchhill', ',', 'and', ',', 'therefore', ',', 'if', 'quite', 'convenient', 'to', 'you', 'and', 'mrs.', 'vernon', 'to', 'receive', 'me', 'at', 'present', ',', 'i', 'shall', 'hope', 'within', 'a', 'few', 'days', 'to', 'be', 'introduced', 'to', 'a', 'sister', 'whom', 'i', 'have', 'so'

## 1.5 Agregar `<s>` y `</s>` y preparar frecuencias de unigramas

Agregaremos `<s>` al inicio y `</s>` al final de cada oracion. Las marcas se agregan tanto al entrenamiento como al test.

Despues de reemplazar los tokens de frecuencia uno por `<UNK>` en entrenamiento, volvemos a contar los tokens. Esta es la frecuencia que usara el modelo, porque ahora todas las apariciones de tokens desconocidos estan agrupadas bajo `<UNK>`. El conteo sigue realizandose con `np.unique`; adicionalmente creamos un diccionario para consultar facilmente la frecuencia de un token.

In [147]:
def agregar_marcas(oraciones):
    """Agrega una marca de inicio y una marca de final a cada oracion.

    Parameters
    ----------
    oraciones: list[list[str]]
        Oraciones tokenizadas.

    Returns
    -------
    list[list[str]]
        Oraciones con `<s>` al inicio y `</s>` al final.
    """
    oraciones_marcadas = []

    for oracion in oraciones:
        nueva_oracion = ["<s>"]

        for token in oracion:
            nueva_oracion.append(token)

        nueva_oracion.append("</s>")
        oraciones_marcadas.append(nueva_oracion)

    return oraciones_marcadas


In [148]:
# Agregación de marcas de inicio y fin a las oraciones
austen_train_marcado = agregar_marcas(austen_train_unk)
twain_train_marcado = agregar_marcas(twain_train_unk)

austen_test_marcado = agregar_marcas(austen_test_unk)
twain_test_marcado = agregar_marcas(twain_test_unk)


In [149]:
# Agregación de marcas de inicio y de fin al vocabulario.

# Las marcas forman parte de los tokens que el modelo puede reconocer.
vocabulario_austen.add("<s>")
vocabulario_austen.add("</s>")
vocabulario_twain.add("<s>")
vocabulario_twain.add("</s>")


In [150]:
# Recontar Austen sobre la representacion final del entrenamiento.
tokens_austen_modelo = []
for oracion in austen_train_marcado:
    for token in oracion:
        tokens_austen_modelo.append(token)

tokens_austen_modelo, frecuencias_austen_modelo = np.unique(
    tokens_austen_modelo,
    return_counts=True
)

frecuencias_austen_modelo_dict = {}
for token, frecuencia in zip(tokens_austen_modelo, frecuencias_austen_modelo):
    frecuencias_austen_modelo_dict[token] = frecuencia

# Recontar Twain sobre la representacion final del entrenamiento.
tokens_twain_modelo = []
for oracion in twain_train_marcado:
    for token in oracion:
        tokens_twain_modelo.append(token)

tokens_twain_modelo, frecuencias_twain_modelo = np.unique(
    tokens_twain_modelo,
    return_counts=True
)

frecuencias_twain_modelo_dict = {}
for token, frecuencia in zip(tokens_twain_modelo, frecuencias_twain_modelo):
    frecuencias_twain_modelo_dict[token] = frecuencia

print("Primeras oraciones Austen con marcas:")
for oracion in austen_train_marcado[:2]:
    print(oracion)

print("\nFrecuencias finales de Austen:")
print("<UNK>:", frecuencias_austen_modelo_dict.get("<UNK>", 0))
print("<s>:", frecuencias_austen_modelo_dict.get("<s>", 0))
print("</s>:", frecuencias_austen_modelo_dict.get("</s>", 0))

Primeras oraciones Austen con marcas:
['<s>', 'lady', 'susan', 'by', 'jane', 'austen', 'contents', 'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii', 'xiii', 'xiv', 'xv', 'xvi', 'xvii', 'xviii', 'xix', 'xx', 'xxi', 'xxii', 'xxiii', 'xxiv', 'xxv', 'xxvi', 'xxvii', 'xxviii', 'xxix', 'xxx', 'xxxi', 'xxxii', 'xxxiii', 'xxxiv', 'xxxv', 'xxxvi', 'xxxvii', 'xxxviii', 'xxxix', 'xl', 'xli', 'conclusion', 'i', '_lady', 'susan', 'vernon', 'to', 'mr.', 'vernon._', 'langford', ',', 'dec.', 'my', 'dear', 'brother', ',', '—', 'i', 'can', 'no', 'longer', 'refuse', 'myself', 'the', 'pleasure', 'of', 'profiting', 'by', 'your', 'kind', 'invitation', 'when', 'we', 'last', 'parted', 'of', 'spending', 'some', 'weeks', 'with', 'you', 'at', 'churchhill', ',', 'and', ',', 'therefore', ',', 'if', 'quite', 'convenient', 'to', 'you', 'and', 'mrs.', 'vernon', 'to', 'receive', 'me', 'at', 'present', ',', 'i', 'shall', 'hope', 'within', 'a', 'few', 'days', 'to', 'be', 'introduced', 'to', 'a', 